# Phase 4 — Exploratory Data Analysis (EDA)

**Project:** EcoFarm Circular — AI-Based Intelligent Energy Management System  
**Datasets:** Real Meteorological & Solar (`2020–2023`, Settat) + Simulated Thermal Subsystems (`simulated_physics`)

## Learning Objectives
1. Verify chronological continuity, duplicate count (`0`), missingness (`0`), and investigate PVGIS midday sentinel zeros (`13` hours) without blind deletion.
2. Analyze diurnal, monthly, and seasonal supply/demand patterns (`DJF` winter drying peak vs. `JJA` summer solar peak).
3. Quantify Pearson and Spearman correlation structures across real and simulated features while maintaining provenance separation and avoiding causal fallacy.
4. Export EDA-ready derived-feature CSVs, correlation matrices, and plots.


In [ ]:
# Google Colab & Portable Environment Bootstrap (Relative Paths + Optional Drive/Upload)
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Using Colab built-in scientific Python stack (pandas, numpy, scikit-learn, matplotlib, seaborn).')

# Option A (Recommended in Colab): Mount Google Drive if EcoFarm_AI is saved on Drive
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust folder name below if stored elsewhere in Google Drive:
    os.chdir('/content/drive/MyDrive/EcoFarm_AI')

# Resolve EcoFarm_AI project root reliably from notebooks/, root, or Colab /content/
cwd = Path.cwd().resolve()
if (cwd / 'src').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / 'EcoFarm_AI' / 'src').is_dir():
    PROJECT_ROOT = cwd / 'EcoFarm_AI'
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment   :', 'Google Colab' if IN_COLAB else 'Local / Jupyter')
print('PROJECT_ROOT  :', PROJECT_ROOT)


In [ ]:
# Dataset & Module Availability Check (with Optional Colab Upload Prompt)
required_files = [
    'data/processed/solar_weather_merged_clean.csv',
    'data/simulated/thermal_sim_medium.csv',
    'reports/eda_metrics_summary.json',
]
missing = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
print(f'Verified {len(required_files) - len(missing)}/{len(required_files)} required project files in {PROJECT_ROOT}.')

if missing:
    print('Missing relative files:', missing)
    if IN_COLAB:
        from google.colab import files
        print('Upload the missing EcoFarm CSV / Phase ZIP files below:')
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(PROJECT_ROOT.parent if PROJECT_ROOT.name == 'EcoFarm_AI' else PROJECT_ROOT)
                print('Extracted uploaded archive:', fname)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import PROCESSED_DATA_DIR, SIMULATED_DATA_DIR, REPORTS_DIR

FIG_DIR = REPORTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

df_real = pd.read_csv(PROCESSED_DATA_DIR / 'solar_weather_merged_clean.csv', parse_dates=['timestamp'])
df_sim = pd.read_csv(SIMULATED_DATA_DIR / 'thermal_sim_medium.csv', parse_dates=['timestamp'])
print('Real shape:', df_real.shape, '| Simulated medium shape:', df_sim.shape)
assert int(df_real.isna().sum().sum()) == 0 and int(df_real['timestamp'].duplicated().sum()) == 0

midday_sentinels = df_real[(df_real['hour_utc'].between(9, 15)) & (df_real['pvgis_sun_elevation_deg'] == 0.0)]
print('Identified PVGIS midday sentinel-zero frames:', len(midday_sentinels))


In [ ]:
# Correlation Analysis (Pearson vs Spearman) & Seasonal Boxplots
real_cols = ['pvgis_irradiance_wm2', 'pvgis_pv_power_kw', 'meteo_ambient_temp_c', 'meteo_relative_humidity_pct', 'meteo_wind_speed_ms']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(df_real[real_cols].corr(method='pearson'), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0])
axes[0].set_title('Pearson Correlation — Real Weather & PV')
sns.heatmap(df_real[real_cols].corr(method='spearman'), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title('Spearman Correlation — Real Weather & PV')
plt.tight_layout(); plt.show()

monthly = df_sim.assign(month=df_sim['timestamp'].dt.month).groupby('month')[['dryer_thermal_demand_kw', 'solar_thermal_power_kw', 'stored_thermal_energy_kwh']].mean()
display(monthly.round(2))


## Conclusions & Limitations
- Irradiance vs. PV power correlation is near-linear (`r = +0.9987`), while dryer active demand anti-correlates strongly with ambient temperature (`r = -0.9959`) and increases with winter relative humidity (`r = +0.8206`).
- Correlation reflects shared solar seasonality or deterministic Phase 3 equations, not empirical pilot-plant causality.


In [ ]:
# Export & Download Phase 4 Package (CSV, XLSX, Notebook, Code, Figures & Reports)
export_dir = PROJECT_ROOT / 'exports' / 'phase_04'
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = export_dir / 'EcoFarm_Phase04_Package.zip'

patterns = ['reports/phase4_eda_findings.md', 'reports/eda_metrics_summary.json', 'reports/figures/0[1-8]_*.svg', 'reports/metrics/eda_*.csv', 'data/outputs/eda_*.csv', 'notebooks/02_data_analysis.ipynb']
added = set()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(PROJECT_ROOT.glob(pat)):
            if fp.is_file() and '__pycache__' not in fp.parts and fp not in added:
                added.add(fp)
                arc = (Path('EcoFarm_AI') / fp.relative_to(PROJECT_ROOT)).as_posix()
                zi = zipfile.ZipInfo(arc, date_time=(2026, 11, 21, 12, 0, 0))
                zi.create_system = 0
                zi.compress_type = zipfile.ZIP_DEFLATED
                zi.external_attr = 0o644 << 16
                zf.writestr(zi, fp.read_bytes())

with zipfile.ZipFile(zip_path, 'r') as zcheck:
    assert zcheck.testzip() is None, 'ZIP CRC check failed!'

print(f'Packaged {len(added)} verified deliverables -> {zip_path} ({zip_path.stat().st_size:,} bytes)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
